# GoProceed prospecting — Wave 7 validation

Reproducible companion analysis for the unified lead base as of **2026-08-24** (Europe/Kyiv). The objective is to expand the pool of construction-project prospects while preserving the previously reviewed 51 leads and avoiding false claims of product demand.

## tl;dr

Wave 7 expands the reviewed base from 1,929 to 2,178 companies. The new search covers additional project-heavy verticals such as fire systems, industrial chimneys, exterior lighting, stormwater drainage, lifts, boiler houses, substations, renewables and industrial automation. Tender awards are used only as workflow/activity signals—not as evidence that a company wants GoProceed.

## Context & Methods

Sources: the existing v7 prospect file, the official Prozorro API-derived Wave 7 supplier set, the normalization summary, browser-verification log and the v8 validation output. Records were filtered to awarded construction/installation work, then deduplicated primarily by EDRPOU. Design-only, expertise, supervision, inventory, survey, research, rental and training records were excluded.

### Key Assumptions

- A Prozorro award confirms legal/project activity, not product interest.
- A future published completion date is a planning signal, not proof of an active site.
- Current product coverage is limited to the seeded N.14/N.15 requirements.
- Other verticals receive a two-week workflow-discovery offer, not a coverage claim.
- Attributed award value is a prioritization signal and is not TAM or revenue.

## Data

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

OUT = Path('/Users/akisliy/Downloads/GoProceed/outputs/01a033d9-c008-7011-bf7b-e1dbd14e2e9d')
def load(name):
    return json.loads((OUT / name).read_text(encoding='utf-8'))

base_v7 = load('prospects_unified_v7_2026-08-24.json')
wave7 = load('prozorro_wave7_supplier_leads_final_2026-08-24.json')
wave7_summary = load('prozorro_wave7_summary_final_2026-08-24.json')
normalization = load('wave7_normalization_summary_2026-08-24.json')
v8 = load('prospects_unified_v8_2026-08-24.json')
validation = load('wave7_v8_validation_2026-08-24.json')
browser_checks = load('wave7_browser_verification_2026-08-24.json')
df = pd.DataFrame(v8)
wave7_df = df[df['lead_id'].astype(str).str.startswith('W7-')].copy()
print({'v7_rows': len(base_v7), 'wave7_supplier_candidates': len(wave7), 'v8_rows': len(df), 'browser_spot_checks': len(browser_checks['companies'])})

{'v7_rows': 1929, 'wave7_supplier_candidates': 325, 'v8_rows': 2178, 'browser_spot_checks': 9}


## Results

In [2]:
lane_counts = df['lane'].value_counts().to_dict()
intent_counts = df['intent_priority'].fillna('—').replace('', '—').value_counts().to_dict()
headline = {
    'total_leads': len(df),
    'unique_new_wave7': normalization['unique_new_supplier_leads'],
    'wave7_overlaps_merged': normalization['overlaps_with_base_1929'],
    'tender_signal_accounts': int((df['tender_count'].fillna(0) > 0).sum()),
    'future_completion_signal_accounts': int((df['live_project_count'].fillna(0) > 0).sum()),
    'high_confidence_accounts': int((df['evidence_confidence'] == 'high').sum()),
    'pilot_now': lane_counts.get('Pilot now', 0),
    'expansion_discovery': lane_counts.get('Expansion discovery', 0),
    'intent_I1': intent_counts.get('I1', 0),
    'intent_I2': intent_counts.get('I2', 0),
}
display(pd.DataFrame([headline]).T.rename(columns={0: 'value'}))

,value
total_leads,2178
unique_new_wave7,249
wave7_overlaps_merged,73
tender_signal_accounts,1919
future_completion_signal_accounts,990
high_confidence_accounts,2112
pilot_now,573
expansion_discovery,1556
intent_I1,127
intent_I2,353


In [3]:
assert validation['ok'] is True
assert len(df) == 2178
assert df['lead_id'].nunique() == len(df)
edrpou = df['edrpou'].fillna('').astype(str).str.replace(r'\D', '', regex=True)
assert edrpou[edrpou.ne('')].nunique() == edrpou.ne('').sum()
previous = df[df['wave'] == 'Перевірено 51 — 24.08']
assert len(previous) == 51
assert (previous['evidence_confidence'] == 'high').all()
assert lane_counts == {'Expansion discovery': 1556, 'Pilot now': 573, 'Requalify': 41, 'Later / requirements': 8}
print('Validation passed: unique IDs/EDRPOU, 51/51 prior leads preserved, lane totals reconcile.')

Validation passed: unique IDs/EDRPOU, 51/51 prior leads preserved, lane totals reconcile.


In [4]:
top_cols = ['lead_id','company_name','edrpou','segment','lane','intent_score','intent_priority','tender_count','live_project_count','domain']
display(wave7_df.sort_values(['intent_score','total_award_value_uah'], ascending=False)[top_cols].head(15).reset_index(drop=True))

,lead_id,company_name,edrpou,segment,lane,intent_score,intent_priority,tender_count,live_project_count,domain
0,W7-001,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ВІК.К...",38119659,Підстанції / силові мережі,Pilot now,89,I1,4,1,
1,W7-002,Товариство з обмеженою відповідальністю «СІТІП...,42356035,Зливова каналізація / дренаж,Requalify,89,I1,4,2,
2,W7-003,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""САН Р...",43187830,Ліфти та ескалатори,Requalify,84,I1,3,1,
3,W7-004,ТОВ Щасливий Сервісбуд,40682121,Зливова каналізація / дренаж,Expansion discovery,83,I1,7,7,
4,W7-005,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ПРОФЕ...",46158743,Зовнішнє освітлення,Requalify,81,I1,4,3,
5,W7-006,"ФОП ""ШЕВЧЕНКО ОЛЕКСАНДР СЕРГІЙОВИЧ""",2836014619,Ліфти та ескалатори,Expansion discovery,80,I1,3,2,
6,W7-007,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""СЕМ-Т...",41008865,Зливова каналізація / дренаж,Expansion discovery,79,I1,7,0,semteh.com
7,W7-008,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ТЕПЛО...",34329599,Промислові котельні,Pilot now,79,I1,14,0,
8,W7-009,"ФОП ""МАКОГОН АНАТОЛІЙ ГРИГОРОВИЧ""",2290000233,Ліфти та ескалатори,Expansion discovery,79,I1,4,1,
9,W7-010,"КП БМР ""МШЕУ""",24223578,Зовнішнє освітлення,Requalify,75,I1,6,2,


In [5]:
lane_table = pd.Series(lane_counts, name='companies').rename_axis('recommended route').reset_index()
lane_table['share'] = (lane_table['companies'] / len(df)).map(lambda value: f'{value:.1%}')
display(lane_table)

,recommended route,companies,share
0,Expansion discovery,1556,71.4%
1,Pilot now,573,26.3%
2,Requalify,41,1.9%
3,Later / requirements,8,0.4%


In [6]:
segment_counts = wave7_df['segment'].value_counts().rename_axis('Wave 7 segment').reset_index(name='new companies')
display(segment_counts.head(15))

,Wave 7 segment,new companies
0,Пожежні системи,46
1,Промислові димові труби,32
2,Зовнішнє освітлення,31
3,Ліфти та ескалатори,24
4,Промислові котельні,24
5,Зливова каналізація / дренаж,24
6,Підстанції / силові мережі,16
7,Промислова електрика / ПНР,8
8,Покрівлі та гідроізоляція,7
9,Теплові насоси / геотермальні системи,6


## Takeaways

1. The lead pool is materially broader than sanitary-technical and electrical installation alone. Wave 7 contributes 249 net-new companies after merging 73 overlaps.
2. The nearest-term motion remains split: 573 accounts can receive an N.14/N.15 pilot message, while 1,556 should receive an honest workflow-discovery message.
3. All prior 51 leads remain present and high confidence.
4. The database is ready for prioritization and outreach, but not for claiming validated demand: the current evidence base still has zero replies, interviews or pilot commitments.